# Lab 01 · Read and query a circuit DAG

**Qiskit Compiler Contribution Lab**  
Independent learning material by Mehar Ali · Python 3.12 / Qiskit 2.6.0

**Goal:** build a reliable helper for finding operations in a circuit, then explain why its tests matter.  
**Prerequisites:** Python functions, classes, iterables, and basic circuit literacy.  
**Material version:** pilot-revision-1-2026-10-22 · revised after first-round feedback; not yet retested with learners.  
**Pilot timing:** allow a 75-minute session plus separate setup; record unfinished attempts at the 40-minute exercise limit. This is a session budget, not a promised completion time.

This is the first step in a general compiler pathway: DAGs → analysis passes → transformations → correctness → regressions → contribution review. No familiarity with the author's PRs is required.

A compiler diagnostic needs to locate measurements and resets together. We will implement its selection helper as a practice exercise. Qiskit already offers operation-selection APIs; this lab does not modify Qiskit or produce a new upstream patch.

## 1 · Use the project kernel

Launch from the folder containing `pyproject.toml` and `uv.lock`:

```sh
uv sync --locked
uv run qclab doctor
uv run python -c "import sys; print(sys.executable)"
uv run jupyter lab notebooks/01-circuit-dag.ipynb
```

These are the same commands as the README; notebook dependencies are included by default. You do not need a separate `--group notebooks` flag or a bare `python` command.

**First check the kernel path below.** It should belong to this project's `.venv`, not system Python or a different copy of the project. A “Python 3” label alone does not prove that you selected the right environment. In VS Code, use the notebook's **Select Kernel** control, not just the terminal interpreter setting. See the [setup checklist](../docs/setup.md) for JupyterLab, VS Code, and WSL instructions.

Run cells top to bottom with **Shift+Enter**. If an import fails, fix the environment before attempting the exercise; do not install packages into an unrelated interpreter. This notebook needs the repository's installed `qclab` helper.

After editing a function, rerun its definition cell and then the checker. Before calling the exercise complete, use **Restart Kernel and Run All Cells**. Passing the checker and completing a clean notebook run are two separate observations. No hardware or IBM account is needed.

In [ ]:
import sys

print("Notebook kernel:", sys.executable)
print("Compare its .venv folder with the terminal command in the setup checklist.")

In [ ]:
import sys

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import Measure, Reset
from qiskit.converters import circuit_to_dag

from qclab.notebook import check_solution

print("Python:", sys.version.split()[0])
print("Qiskit:", qiskit.__version__)
print("Filter classes:", Measure.__name__, Reset.__name__)
print("Kernel:", sys.executable)
assert qiskit.__version__ == "2.6.0", "Select the project's pinned environment."

## 2 · Inspect a real circuit

The `h` and `cx` operations are gates. `measure` records a qubit outcome into a classical bit; `reset` prepares a qubit in its zero state. The diagnostic needs to locate both measurements and resets, including repeated occurrences.

Run the circuit cell. Predict which operations a filter for `[Measure, Reset]` should return before reading the node listing.

In [ ]:
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure(1, 0)
circuit.reset(1)
circuit.measure(1, 1)
circuit.draw(output="text")

**Your prediction:** replace this sentence with the operation names you expect, in order. Which classical bit does each measurement write to?

## 3 · Read the DAG

A directed acyclic graph represents dependencies between instructions. It contains input and output nodes as well as operation nodes. Here we use `dag.op_nodes()` to inspect only operations.

Each operation node has `op`, `qargs`, and `cargs`. An operation type can occur many times on different wires. Selecting a type should keep every matching node.

This lab preserves the order returned by `op_nodes()`; it does not assume that order is a topological traversal. For algorithms that require dependency order, investigate `topological_op_nodes()` separately.

In [ ]:
dag = circuit_to_dag(circuit)
for node in dag.op_nodes():
    qubits = [dag.find_bit(bit).index for bit in node.qargs]
    clbits = [dag.find_bit(bit).index for bit in node.cargs]
    print(f"{node.name:8} qubits={qubits}, clbits={clbits}")

### Before the exercise: a generator is a one-way iterator

Predict the second list printed below. Iterating a generator advances it; it does not restart itself for the next circuit node. A tuple can be read repeatedly.

The lesson is to consume an iterable **once per call**, before visiting the nodes—not once per node. Start with a list, then a tuple, then a generator. All three should describe the same requested types.

In [ ]:
one_way = (kind for kind in (Measure, Reset))
first_read = list(one_way)
second_read = list(one_way)
print("First read:", [kind.__name__ for kind in first_read])
print("Second read:", second_read)
assert first_read == [Measure, Reset]
assert second_read == []

fresh_generator = (kind for kind in (Measure, Reset))
reusable_types = tuple(fresh_generator)
assert list(reusable_types) == [Measure, Reset]
assert list(reusable_types) == [Measure, Reset]
print("Stored tuple can be read again:", [kind.__name__ for kind in reusable_types])

### Validate the request before looking for matches

There are two separate questions: **is this filter valid?** and **which nodes match it?** An empty circuit answers only the second question.

| Filter | Empty DAG | DAG with a measurement |
|---|---|---|
| `[]` | Return `[]` | Return `[]` |
| `[Measure]` | Return `[]` | Select its measurements |
| `[Measure, "reset"]` | Raise `TypeError` | Raise `TypeError` |

`Measure` and `Reset` are classes; `"reset"` is a string, not a class. Finding an early match does not make an invalid later entry valid. Validate **every entry before matching**, even if there are no operations.

The next cell demonstrates input validation alone—not the completed selector. The expected error is caught so the notebook can continue.

In [ ]:
empty_example = circuit_to_dag(QuantumCircuit(1))
assert list(empty_example.op_nodes()) == []

invalid_filter = [Measure, "reset"]
try:
    for entry in invalid_filter:
        if not isinstance(entry, type):
            raise TypeError("Every filter entry must be a Python class")
except TypeError as error:
    print("Expected TypeError, even with an empty DAG:", error)
else:
    raise AssertionError("The invalid filter should have been rejected")

## 4 · Implement the selection helper

Edit the next cell. Preserve existing behavior while adding iterable inputs. Work in three stages: normalize the filter once, validate every entry, then visit each node once. Keep `None` (select all) distinct from an empty filter (select none).

| Input | Required behavior |
|---|---|
| `None` | All operation nodes |
| `Measure` | Every measurement |
| `[Measure, Reset]`, a tuple, or a set | Every matching node, once |
| A generator yielding classes | Same selection for every node |
| Empty iterable | Empty list |
| A superclass such as `Instruction` | Match subclasses using `isinstance` |
| Non-iterable such as `42` | Raise `TypeError` |
| Any iterable entry that is not a class | Raise `TypeError`, even on an empty DAG |

Return a **list of original operation nodes**, preserve their relative `dag.op_nodes()` order, and do not modify the DAG. Repeated filter types must not duplicate nodes.

Start by running the existing single-type implementation. Then make a list of two types work, followed by generators and invalid inputs.

In [ ]:
def select_op_nodes(dag, operation_types=None):
    """Return original matching operation nodes, without modifying the DAG.

    Accept None, a class, or an iterable of classes.
    Preserve the relative order of dag.op_nodes().
    Empty filters match nothing; invalid filter entries raise TypeError.
    """
    # This starter supports None and one class. Extend it to handle iterables.
    if operation_types is None:
        return list(dag.op_nodes())
    if not isinstance(operation_types, type):
        raise TypeError("operation_types must be a class")
    return [node for node in dag.op_nodes() if isinstance(node.op, operation_types)]

In [ ]:
measurement_nodes = select_op_nodes(dag, Measure)
print([node.name for node in measurement_nodes])
assert [node.name for node in measurement_nodes] == ["measure", "measure"]

## 5 · Run the acceptance checks

The next cell checks **your current function** against the same 28 cases used by the command-line lab. The unfinished starter is expected to fail some cases.

The result is captured as text, so an expected failure will not stop the notebook. Read the first named failure, edit the function, rerun its cell, and rerun this cell. A green result verifies the behavior contract; your independent test and explanation below are separate learning outcomes.

Each check runs in a fresh process using a temporary snapshot of your function. It does not overwrite a solution file and stops after 30 seconds if the solution hangs.

In [ ]:
result = check_solution(select_op_nodes)
print("Acceptance status:", "PASS" if result.passed else "NOT YET")
if not result.passed:
    print("Next action: read the first named failure below; work on one behavior at a time.")
    print("Rerun the solution definition, then this cell. NOT YET is expected for the starter.")
print(result.output)

## Hints · open only when needed

<details>
<summary>Hint 1: normalize the input once</summary>

A single class can become a one-element tuple. An iterable can be materialized once. Keep `None` distinct from an empty filter.
</details>

<details>
<summary>Hint 2: why does a generator behave differently?</summary>

A generator is consumed as you iterate. If the filter is consumed separately for each node, the first node can exhaust it and later nodes see no types. Materialize it before visiting nodes.
</details>

<details>
<summary>Hint 3: validate before matching</summary>

`isinstance` can short-circuit when an early class matches, hiding an invalid later entry. Validate all entries first—even if there are no nodes. Use subclass-aware matching: some Qiskit standard gates are singleton subclasses.
</details>

<details>
<summary>Hint 4: preserve nodes, not filter order</summary>

Visit each node once and ask whether any requested type matches it. Looping over the types first can reorder or duplicate the results.
</details>

## 6 · Write an independent regression test

Create a small circuit with a gate, two measurements, and a reset. Pass a generator of types. Assert the expected operation names using a literal expected list.

Before running it, explain how a solution that consumes the generator inside the node loop would fail. You may change the circuit and expected values; make the test demonstrate your reasoning.

Temporarily introduce the mistake your test is intended to catch. Rerun the definition and your test, and confirm it fails for the predicted reason. Restore your solution, rerun its definition, and confirm the test passes. A passing assertion alone does not demonstrate that it would catch the bug.

In [ ]:
# Write and run your independent regression test here.

## 7 · Explain your change

Replace the prompts below with your own answers:

1. **User problem:** what does the caller need?
2. **Cause:** why can a generator pass for one node and fail for another?
3. **Validation:** why must `[Measure, "reset"]` fail even if a measurement matches?
4. **Compatibility:** why is exact class identity too restrictive?
5. **Evidence:** what failed before, and what now passes?
6. **Limit:** how is this exercise different from changing Qiskit's compiler itself?

Ask another person to review your test and explanation. You should be able to explain the code without reading the instructor notebook.

## Finish with a clean run

Save your work. Choose **Restart Kernel and Run All Cells** and verify that the acceptance status is **PASS** and your own regression test passes.

After your attempt, you can compare with [the instructor notebook](instructor/01-circuit-dag-solution.ipynb). It includes a complete solution and facilitator notes. This is open-book practice, not a secure exam.

Continue with [Lab 02: Analyze without changing](02-analysis-pass.ipynb), a notebook on analysis passes and circuit reports. See the [curriculum roadmap](../docs/roadmap.md) for later planned labs.

Before any real upstream submission, read the [upstream transition guide](../docs/upstream.md), including Qiskit's current contribution and AI-use policies. This practice exercise is not a patch to submit upstream.

## Primary references

- [IBM: DAGCircuit](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.dagcircuit.DAGCircuit)
- [IBM: circuit converters](https://quantum.cloud.ibm.com/docs/en/api/qiskit/converters)
- [Python: isinstance](https://docs.python.org/3/library/functions.html#isinstance)
- [Qiskit: contribution guide](https://github.com/Qiskit/qiskit/blob/main/CONTRIBUTING.md)

Independent project; not affiliated with or endorsed by IBM, Qiskit, or Unitary Foundation. Original material is MIT licensed. The scaffold was developed with AI assistance.